In [ ]:
!pip install openmeteo_requests
!pip install pandas
!pip install requests_cache
!pip install retry_requests

import openmeteo_requests
import os
import pandas as pd
import requests_cache
from retry_requests import retry
import time
import sys
import shutil
from pathlib import Path

try:
	# Setup the Open-Meteo API client with cache and retry on error
	cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
	retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
	openmeteo = openmeteo_requests.Client(session = retry_session)

	# Make sure all required weather variables are listed here
	# The order of variables in hourly or daily is important to assign them correctly below
	url = "https://archive-api.open-meteo.com/v1/archive"
except Exception as e:
	print(e)
	input()
	sys.exit()

file = max(Path(".").glob("*.csv"), key=lambda f: f.stat().st_mtime)

df = pd.read_csv(file)

number = int(file.name.split("_")[0])

os.makedirs(f"municipios_meteo", exist_ok=True)

c = 0
rows_lat = []
rows_lon = []
for _, row in df.iterrows():
	rows_lat.append(row["LAT"])
	rows_lon.append(row["LON"])

params = {
	"latitude": [rows_lat],
	"longitude": [rows_lon],
	"start_date": "2000-01-01",
	"end_date": "2024-12-31",
	"daily": ["rain_sum", "shortwave_radiation_sum", "relative_humidity_2m_mean", "soil_moisture_0_to_100cm_mean", "soil_temperature_0_to_100cm_mean", "temperature_2m_mean"],
	"timezone": "America/Sao_Paulo",
}

try:
	responses = openmeteo.weather_api(url, params = params)
	#response = responses[0]
except Exception as e:
	print(e)
	#input("Pressione qualquer tecla para sair...")
	#continue

for response in responses:
	municipio = df["MUN"][c]
	c+=1
	print(f"{c}")
	print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Município: {municipio}\n")

	# Process daily data. The order of variables needs to be the same as requested.
	daily = response.Daily()
	daily_rain_sum = daily.Variables(0).ValuesAsNumpy()
	daily_shortwave_radiation_sum = daily.Variables(1).ValuesAsNumpy()
	daily_relative_humidity_2m_mean = daily.Variables(2).ValuesAsNumpy()
	daily_soil_moisture_0_to_100cm_mean = daily.Variables(3).ValuesAsNumpy()
	daily_soil_temperature_0_to_100cm_mean = daily.Variables(4).ValuesAsNumpy()
	daily_temperature_2m_mean = daily.Variables(5).ValuesAsNumpy()

	daily_data = {
		"date": pd.date_range(
			start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
			end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
			freq = pd.Timedelta(seconds = daily.Interval()),
			inclusive = "left"
		).tz_convert(response.Timezone().decode())
	}

	daily_data["rain_sum"] = daily_rain_sum
	daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum
	daily_data["relative_humidity_2m_mean"] = daily_relative_humidity_2m_mean
	daily_data["soil_moisture_0_to_100cm_mean"] = daily_soil_moisture_0_to_100cm_mean
	daily_data["soil_temperature_0_to_100cm_mean"] = daily_soil_temperature_0_to_100cm_mean
	daily_data["temperature_2m_mean"] = daily_temperature_2m_mean

	daily_dataframe = pd.DataFrame(data = daily_data)
	daily_dataframe.to_csv(f"municipios_meteo/{municipio}.csv", index=False)
	#time.sleep(65)

shutil.make_archive(f"{number}_mun-meteo", "zip", "municipios_meteo")